In [ ]:
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
baseline = pypsa.Network(
    "../../resources/network/Historical_2015_reduced_solved.nc"
)

print("Snapshots:", len(baseline.snapshots))
print("Start:", baseline.snapshots[0])
print("End:", baseline.snapshots[-1])

INFO:pypsa.network.io:New version 1.2.4 available! (Current: 1.0.7)
INFO:pypsa.network.io:Imported network 'Historical_2015_reduced (Full)' has buses, carriers, generators, lines, links, loads, storage_units, sub_networks


Snapshots: 168
Start: 2015-01-01 00:00:00
End: 2015-01-07 23:00:00


In [3]:
bess_case = baseline.copy()

In [4]:
bess_case.add(
    "StorageUnit",
    "Research_BESS_Beauly",
    bus="Beauly",
    carrier="Battery",
    
    # Power capacity
    p_nom=100,
    
    # 100 MW × 2 hours = 200 MWh
    max_hours=2,
    
    # Battery efficiencies
    efficiency_store=0.92,
    efficiency_dispatch=0.92,
    
    # 0.1% loss per hour
    standing_loss=0.001,
    
    # Start empty
    state_of_charge_initial=0,
    
    # Fixed-size battery
    p_nom_extendable=False,
    
    # Small operating cost
    marginal_cost=0
)

In [5]:
state_of_charge_initial=0

In [6]:
bess_case.storage_units.loc["Research_BESS_Beauly"]

bus                                    Beauly
control                                    PQ
type                                         
p_nom                                   100.0
p_nom_mod                                 0.0
p_nom_extendable                        False
p_nom_min                                 0.0
p_nom_max                                 inf
p_nom_set                                 NaN
p_min_pu                                 -1.0
p_max_pu                                  1.0
p_set                                     NaN
q_set                                     0.0
p_dispatch_set                            NaN
p_store_set                               NaN
sign                                      1.0
carrier                               Battery
spill_cost                                0.0
marginal_cost                             0.0
marginal_cost_quadratic                   0.0
marginal_cost_storage                     0.0
capital_cost                      

In [7]:
print(
    "Battery energy capacity:",
    bess_case.storage_units.loc[
        "Research_BESS_Beauly", "p_nom"
    ]
    *
    bess_case.storage_units.loc[
        "Research_BESS_Beauly", "max_hours"
    ],
    "MWh"
)

Battery energy capacity: 200.0 MWh


In [8]:
status, condition = bess_case.optimize(
    solver_name="highs"
)

print("Status:", status)
print("Condition:", condition)

Index(['16'], dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 6/6 [00:00<00:00,  7.59it/s]
INFO:linopy.io: Writing time: 4.11s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 310800 primals, 641592 duals
Objective: 2.01e+08
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper, Link-fix-p-lower, Link-fix-p-upper, Link-p_set, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state_of_charge-lower, StorageUnit-fix-state_of_charge-upper, Kirchhoff-Voltage-Law, StorageUnit-energy_balance were not assigned to the network.


Status: ok
Condition: optimal


In [9]:
bess_name = "Research_BESS_Beauly"

bess_charge = bess_case.storage_units_t.p_store[bess_name]
bess_discharge = bess_case.storage_units_t.p_dispatch[bess_name]
bess_soc = bess_case.storage_units_t.state_of_charge[bess_name]

print("Total charging:", round(bess_charge.sum(), 1), "MWh")
print("Total discharging:", round(bess_discharge.sum(), 1), "MWh")
print("Maximum SOC:", round(bess_soc.max(), 1), "MWh")
print("Final SOC:", round(bess_soc.iloc[-1], 1), "MWh")

Total charging: 947.0 MWh
Total discharging: 797.2 MWh
Maximum SOC: 200.0 MWh
Final SOC: -0.0 MWh


In [10]:
def calculate_beauly_curtailment(network):

    wind = network.generators[
        (network.generators["bus"] == "Beauly")
        & (network.generators["carrier"] == "wind_onshore")
    ]

    wind_names = wind.index

    available = (
        network.generators_t.p_max_pu[wind_names]
        .mul(wind["p_nom"], axis=1)
        .sum(axis=1)
    )

    dispatched = (
        network.generators_t.p[wind_names]
        .sum(axis=1)
    )

    curtailed = (available - dispatched).clip(lower=0)
    curtailed[curtailed < 1e-6] = 0

    return {
        "available_MWh": available.sum(),
        "dispatched_MWh": dispatched.sum(),
        "curtailed_MWh": curtailed.sum(),
        "curtailment_rate_pct":
            100 * curtailed.sum() / available.sum()
    }


baseline_result = calculate_beauly_curtailment(baseline)
bess_result = calculate_beauly_curtailment(bess_case)

comparison = pd.DataFrame({
    "Baseline": baseline_result,
    "100MW_200MWh_BESS": bess_result
})

comparison

,Baseline,100MW_200MWh_BESS
available_MWh,97748.675085,97748.675085
dispatched_MWh,95758.124867,92316.251604
curtailed_MWh,1990.550218,5432.423481
curtailment_rate_pct,2.036396,5.557542


In [11]:
curtailment_reduction = (
    baseline_result["curtailed_MWh"]
    - bess_result["curtailed_MWh"]
)

reduction_pct = (
    100
    * curtailment_reduction
    / baseline_result["curtailed_MWh"]
)

print(
    "Curtailment reduction:",
    round(curtailment_reduction, 1),
    "MWh"
)

print(
    "Curtailment reduction:",
    round(reduction_pct, 2),
    "%"
)

Curtailment reduction: -3441.9 MWh
Curtailment reduction: -172.91 %


In [12]:
# Create a control case with NO additional BESS
control_case = pypsa.Network(
    "../../resources/network/Historical_2015_reduced_solved.nc"
)

# Re-optimise using EXACTLY the same method used for our BESS case
status, condition = control_case.optimize(
    solver_name="highs"
)

print("Status:", status)
print("Condition:", condition)

INFO:pypsa.network.io:New version 1.2.4 available! (Current: 1.0.7)
INFO:pypsa.network.io:Imported network 'Historical_2015_reduced (Full)' has buses, carriers, generators, lines, links, loads, storage_units, sub_networks
Index(['16'], dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 6/6 [00:00<00:00,  9.07it/s]
INFO:linopy.io: Writing time: 3.96s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 310296 primals, 640416 duals
Objective: 2.01e+08
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Line-fix-s-lower, Line-fix-s-upper, Link-fix-p-lower, Link-fix-p-upper, Link-p_set, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, Storage

Status: ok
Condition: optimal


In [13]:
control_result = calculate_beauly_curtailment(control_case)

check = pd.DataFrame({
    "Original_PyPSA_GB": baseline_result,
    "Reoptimised_Control": control_result,
    "100MW_200MWh_BESS": bess_result
})

check

,Original_PyPSA_GB,Reoptimised_Control,100MW_200MWh_BESS
available_MWh,97748.675085,97748.675085,97748.675085
dispatched_MWh,95758.124867,92131.735142,92316.251604
curtailed_MWh,1990.550218,5616.939943,5432.423481
curtailment_rate_pct,2.036396,5.746308,5.557542


In [14]:
from pathlib import Path

network_files = list(
    Path("../../resources/network").glob(
        "Historical_2015_reduced*.nc"
    )
)

for file in network_files:
    print(file.name)

Historical_2015_reduced.nc
Historical_2015_reduced_network.nc
Historical_2015_reduced_network_demand_renewables_thermal_generators_storage_hydrogen_interconnectors.nc
Historical_2015_reduced_solved.nc


## 5. Controlled BESS Counterfactual

Both the baseline and BESS scenarios are constructed from the same fully assembled
pre-solve PyPSA-GB network. This ensures that differences in results arise from the
addition of storage rather than from differences in optimisation workflow.

In [15]:
import pypsa
import pandas as pd
import numpy as np

input_network_path = (
    "../../resources/network/"
    "Historical_2015_reduced_network_demand_renewables_"
    "thermal_generators_storage_hydrogen_interconnectors.nc"
)

prepared_network = pypsa.Network(input_network_path)

print("Snapshots:", len(prepared_network.snapshots))
print("Buses:", len(prepared_network.buses))
print("Generators:", len(prepared_network.generators))
print("Storage units:", len(prepared_network.storage_units))
print("Links:", len(prepared_network.links))

INFO:pypsa.network.io:New version 1.2.4 available! (Current: 1.0.7)
INFO:pypsa.network.io:Imported network 'Reduced Network' has buses, carriers, generators, lines, links, loads, storage_units


Snapshots: 8760
Buses: 34
Generators: 1721
Storage units: 7
Links: 6


In [18]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\nSearching for solve_network.py...")

for path in Path("../..").rglob("solve_network.py"):
    print(path.resolve())

Current working directory:
c:\Users\shubh\PyPSA-GB\project1_gb_market\notebooks

Searching for solve_network.py...
C:\Users\shubh\PyPSA-GB\scripts\solve\solve_network.py


In [21]:
# Keep only the same 7-day period used in our original experiment
prepared_network.set_snapshots(
    prepared_network.snapshots[
        (prepared_network.snapshots >= "2015-01-01 00:00:00")
        & (prepared_network.snapshots <= "2015-01-07 23:00:00")
    ]
)

print("Snapshots after trimming:", len(prepared_network.snapshots))
print("Start:", prepared_network.snapshots[0])
print("End:", prepared_network.snapshots[-1])

Snapshots after trimming: 168
Start: 2015-01-01 00:00:00
End: 2015-01-07 23:00:00


In [23]:
import inspect
import scripts.solve.solve_network as solve_module

functions = [
    name
    for name, obj in inspect.getmembers(solve_module, inspect.isfunction)
]

print(functions)

['_align_neso_boundary_limit_series', '_apply_csv_outage_schedule', '_apply_neso_boundary_limits', '_apply_synthetic_outage_schedule', '_build_neso_boundary_constraints_callback', '_fill_missing_neso_limits', '_get_neso_constraint_mode', '_inspect_neso_boundary_coverage', '_load_cached_boundary_file', '_load_neso_boundary_inputs', 'apply_line_rating_overrides', 'apply_load_shedding_limits', 'apply_outage_schedule', 'apply_transmission_relaxation', 'build_hydro_constraints_callback', 'combine_extra_functionalities', 'configure_solver', 'export_optimization_results', 'generate_optimization_summary', 'get_solve_mode_from_config', 'improve_numerical_conditioning', 'load_network', 'log_hydro_constraint_setup', 'save_network', 'setup_logging', 'validate_network_costs']


In [24]:
from pathlib import Path

solve_file = Path("../../scripts/solve/solve_network.py")

lines = solve_file.read_text(encoding="utf-8").splitlines()

# Show the final 120 lines of the actual PyPSA-GB solver script
for i, line in enumerate(lines[-120:], start=len(lines)-119):
    print(f"{i}: {line}")

1951:                 n_must_run = (network.generators.p_min_pu > 0).sum()
1952:                 logger.info(f"Generators with p_min_pu > 0: {n_must_run}")
1953:                 logger.info("These generators (e.g. nuclear, biomass) will be forced to run if available")
1954:             else:
1955:                 logger.info("No 'p_min_pu' column found - all generators can turn off")
1956:         logger.info("=" * 80)
1957:         
1958:         # Check generator aggregation status
1959:         if hasattr(network, 'meta') and network.meta.get('aggregated', False):
1960:             logger.info("NOTE: Network uses GENERATOR AGGREGATION")
1961:             logger.info("  Generators of same carrier at same bus are aggregated")
1962:             logger.info("  This significantly improves solve speed")
1963: 
1964:         
1965:         # Configure solver
1966:         solver_name, solver_options = configure_solver(network, solver_name, solver_options, logger)
1967:         
1968:      

In [25]:
# Find the exact optimisation call inside PyPSA-GB
for i, line in enumerate(lines, start=1):
    if ".optimize(" in line:
        start = max(1, i - 15)
        end = min(len(lines), i + 25)

        print(f"\n--- Optimisation call around line {i} ---\n")

        for j in range(start, end + 1):
            print(f"{j}: {lines[j-1]}")


--- Optimisation call around line 1982 ---

1967:         
1968:         # Run optimization
1969:         logger.info("Starting optimization...")
1970:         logger.info(f"This may take several minutes depending on network size and solver...")
1971:         
1972:         solve_start = time.time()
1973:         
1974:         # Note: skip_objective and track_iterations are PyPSA parameters, NOT solver options
1975:         # They should not be passed in solver_options to avoid Gurobi errors
1976:         log_hydro_constraint_setup(network, scenario_config, logger)
1977:         hydro_callback = build_hydro_constraints_callback(network, scenario_config)
1978:         neso_boundary_callback = _build_neso_boundary_constraints_callback(
1979:             network, scenario_config, logger
1980:         )
1981: 
1982:         status, termination_condition = network.optimize(
1983:             solver_name=solver_name,
1984:             solver_options=solver_options,
1985:             extra_

In [26]:
# Find where scenario_config is created in the actual PyPSA-GB solver

for i, line in enumerate(lines, start=1):
    if "scenario_config =" in line:
        print(f"\n--- scenario_config around line {i} ---\n")

        start = max(1, i - 12)
        end = min(len(lines), i + 15)

        for j in range(start, end + 1):
            print(f"{j}: {lines[j-1]}")


--- scenario_config around line 1782 ---

1770:     logger.info("SOLVING NETWORK WITH PYPSA OPTIMIZATION")
1771:     logger.info("=" * 80)
1772:     
1773:     start_time = time.time()
1774:     
1775:     try:
1776:         # Load network
1777:         input_path = snakemake.input.network
1778:         logger.info(f"Loading network from: {input_path}")
1779:         network = load_network(input_path, custom_logger=logger)
1780:         
1781:         # Get parameters
1782:         scenario_config = snakemake.params.scenario_config
1783:         solver_name = snakemake.params.solver
1784:         solver_options = snakemake.params.solver_options
1785:         scenario_id = scenario_config.get('scenario_id', snakemake.wildcards.scenario)
1786:         
1787:         logger.info(f"Scenario: {scenario_id}")
1788:         logger.info(f"Network loaded: {len(network.buses)} buses, {len(network.generators)} generators")
1789:         logger.info(f"Full year snapshots: {len(network.snapshots)}

In [27]:
# Show only the important setup calls made before network.optimize()

important_terms = [
    "validate_network_costs",
    "apply_transmission_relaxation",
    "apply_line_rating_overrides",
    "apply_load_shedding_limits",
    "apply_outage_schedule",
    "improve_numerical_conditioning",
    "build_hydro_constraints_callback",
    "_build_neso_boundary_constraints_callback",
    "configure_solver",
    "set_snapshots"
]

for i in range(1791, 1989):
    line = lines[i - 1]

    if any(term in line for term in important_terms):
        print(f"{i}: {line}")

1793:         validate_network_costs(network, logger)
1797:         apply_transmission_relaxation(network, scenario_config, logger)
1800:         apply_line_rating_overrides(network, scenario_config, logger)
1804:         apply_outage_schedule(network, scenario_config, logger)
1808:         improve_numerical_conditioning(network, logger)
1881:             network.set_snapshots(selected_snapshots)
1890:         apply_load_shedding_limits(network, logger)
1966:         solver_name, solver_options = configure_solver(network, solver_name, solver_options, logger)
1977:         hydro_callback = build_hydro_constraints_callback(network, scenario_config)
1978:         neso_boundary_callback = _build_neso_boundary_constraints_callback(


In [28]:
import yaml
from pathlib import Path

scenario_file = Path("../../config/scenarios.yaml")

with open(scenario_file, "r", encoding="utf-8") as f:
    scenarios = yaml.safe_load(f)

print(type(scenarios))
print(scenarios.keys())

<class 'dict'>
dict_keys(['Historical_2020_reduced', 'Historical_2020_zonal', 'Historical_2020_ETYS', 'Historical_2020_ETYS_market', 'Historical_2019_reduced', 'Historical_2019_zonal', 'Historical_2019_ETYS', 'Historical_2023_etys', 'Historical_2020_ETYS_6h', 'Historical_2020_reduced_6h', 'Historical_2017_reduced', 'Historical_2015_reduced', 'Historical_2010_reduced', 'Historical_2011_reduced', 'Historical_2012_reduced', 'Historical_2013_reduced', 'Historical_2014_reduced', 'Historical_2016_reduced', 'Historical_2018_reduced', 'Historical_2020_clustered', 'Historical_2019_clustered', 'HT35', 'HT35_flex', 'HT30_event_response', 'HT35_year', 'HT35_without_network_upgrades', 'HT35_reduced', 'HT35_reduced_eload', 'HT35_reduced_desstinee', 'HT50_reduced_eload_2050', 'HT35_zonal', 'HT35_clustered_gsp', 'HT35_clustered', 'HT50_clustered_gsp', 'EE50_clustered', 'HT35_clustered_kmeans', 'HT35_with_disaggregation', 'Test_Historical_2015', 'Test_Future_2030', 'Test_Clustered_GSP', 'Test_Historica

In [29]:
scenario_config = scenarios["Historical_2015_reduced"].copy()

# Add scenario ID because the solver script expects it
scenario_config["scenario_id"] = "Historical_2015_reduced"

print(
    yaml.safe_dump(
        scenario_config,
        sort_keys=False
    )
)

description: 2015 Historical - Reduced Network
modelled_year: 2015
renewables_year: 2015
demand_year: 2015
network_model: Reduced
solve_period:
  enabled: true
  start: 2015-01-01 00:00
  end: 2015-01-07 23:00
scenario_id: Historical_2015_reduced



In [30]:
import logging
from scripts.solve import solve_network as solve_mod

# Simple logger for our research notebook
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("project1_bess")


def solve_research_case(input_network, scenario_config):
    """
    Solve a research case using the same main preprocessing
    and optimisation steps used by PyPSA-GB.
    """

    # Always work on a fresh copy
    n = input_network.copy()

    # Same preprocessing sequence as PyPSA-GB
    solve_mod.validate_network_costs(n, logger)

    solve_mod.apply_transmission_relaxation(
        n, scenario_config, logger
    )

    solve_mod.apply_line_rating_overrides(
        n, scenario_config, logger
    )

    solve_mod.apply_outage_schedule(
        n, scenario_config, logger
    )

    solve_mod.improve_numerical_conditioning(
        n, logger
    )

    solve_mod.apply_load_shedding_limits(
        n, logger
    )

    # Configure HiGHS
    solver_name, solver_options = solve_mod.configure_solver(
        n,
        "highs",
        {"threads": 4},
        logger
    )

    # PyPSA-GB additional constraints
    solve_mod.log_hydro_constraint_setup(
        n, scenario_config, logger
    )

    hydro_callback = (
        solve_mod.build_hydro_constraints_callback(
            n, scenario_config
        )
    )

    neso_callback = (
        solve_mod._build_neso_boundary_constraints_callback(
            n, scenario_config, logger
        )
    )

    extra_functionality = (
        solve_mod.combine_extra_functionalities(
            hydro_callback,
            neso_callback
        )
    )

    # Solve
    status, condition = n.optimize(
        solver_name=solver_name,
        solver_options=solver_options,
        extra_functionality=extra_functionality
    )

    print("Status:", status)
    print("Condition:", condition)
    print("Objective:", n.objective)

    return n

In [32]:
baseline_clean = solve_research_case(
    prepared_network,
    scenario_config
)

INFO:project1_bess:================================================================================
INFO:project1_bess:VALIDATING NETWORK MARGINAL COSTS
INFO:project1_bess:================================================================================
INFO:project1_bess:[OK] Load shedding: 29 units @ £6,000-£6,000/MWh
INFO:project1_bess:[OK] Thermal generators: 171 units, MC: £70.53/MWh (£56.30-£96.65)
INFO:project1_bess:
Marginal cost distribution:
INFO:project1_bess:  Total generators: 1721
INFO:project1_bess:  Zero cost: 978 (56.8%)
INFO:project1_bess:  Positive cost: 743 (43.2%)
INFO:project1_bess:[OK] Fixed-flow interconnector supply: 6 EU_import generators at zero cost (4,972 MW). This is expected for historical fixed-link flows.
INFO:project1_bess:================================================================================
INFO:project1_bess:[OK] NETWORK COST VALIDATION PASSED
INFO:project1_bess:===============================================================================

Status: ok
Condition: unknown
Objective: 0.0


In [33]:
baseline_clean_result = calculate_beauly_curtailment(
    baseline_clean
)

baseline_clean_result

{'available_MWh': 97748.6750851832,
 'dispatched_MWh': 0.0,
 'curtailed_MWh': 97748.6750851832,
 'curtailment_rate_pct': 99.99999999999999}

In [34]:
scenario_names = [
    "Historical_2015_reduced",
    "Historical_2017_reduced",
    "Historical_2019_reduced",
    "Historical_2020_reduced",
    "Historical_2023_etys",
    "Historical_2024_fullyear",
]

for name in scenario_names:
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    print(
        yaml.safe_dump(
            scenarios[name],
            sort_keys=False
        )
    )


Historical_2015_reduced
description: 2015 Historical - Reduced Network
modelled_year: 2015
renewables_year: 2015
demand_year: 2015
network_model: Reduced
solve_period:
  enabled: true
  start: 2015-01-01 00:00
  end: 2015-01-07 23:00


Historical_2017_reduced
description: 2017 Historical - Reduced Network
modelled_year: 2017
renewables_year: 2017
demand_year: 2017
network_model: Reduced
solve_period:
  enabled: true
  start: 2017-01-01 00:00
  end: 2017-01-07 23:00


Historical_2019_reduced
description: 2019 Historical - Reduced Network
modelled_year: 2019
renewables_year: 2019
demand_year: 2019
network_model: Reduced
solve_period:
  enabled: true
  start: 2019-01-01 00:00
  end: 2019-01-07 23:00


Historical_2020_reduced
description: 2020 Historical - Reduced Network (32 buses)
modelled_year: 2020
renewables_year: 2020
demand_year: 2020
network_model: Reduced
solve_period:
  enabled: true
  start: 2020-01-01 00:00
  end: 2020-01-07 23:00


Historical_2023_etys
description: 2023 Histo

In [35]:
import pypsa

jan_2015 = pypsa.Network(
    "../../resources/network/Research_2015_Jan_solved.nc"
)

print("Scenario:", jan_2015.name)
print("Snapshots:", len(jan_2015.snapshots))
print("Start:", jan_2015.snapshots[0])
print("End:", jan_2015.snapshots[-1])
print("Objective:", jan_2015.objective)

Scenario: Research_2015_Jan (Full)
Snapshots: 744
Start: 2015-01-01 00:00:00
End: 2015-01-31 23:00:00
Objective: 970267131.6080409


In [36]:
# Find all wind-related generator carriers in the January model

wind_carriers = [
    carrier
    for carrier in jan_2015.generators["carrier"].dropna().unique()
    if "wind" in carrier.lower()
]

print("Wind carriers found:")
print(wind_carriers)

print("\nInstalled wind capacity by carrier:")

wind_capacity = (
    jan_2015.generators[
        jan_2015.generators["carrier"].isin(wind_carriers)
    ]
    .groupby("carrier")["p_nom"]
    .sum()
    .sort_values(ascending=False)
)

print(wind_capacity)

Wind carriers found:
['wind_onshore', 'wind_offshore', 'embedded_wind']

Installed wind capacity by carrier:
carrier
wind_onshore     7496.75
wind_offshore    4039.40
embedded_wind    3682.00
Name: p_nom, dtype: float64


In [37]:
# Select every wind generator in the GB model
gb_wind = jan_2015.generators[
    jan_2015.generators["carrier"].isin(wind_carriers)
]

gb_wind_names = gb_wind.index

# Maximum wind that could have generated each hour
gb_wind_available = (
    jan_2015.generators_t.p_max_pu[gb_wind_names]
    .mul(gb_wind["p_nom"], axis=1)
    .sum(axis=1)
)

# Wind actually dispatched by the optimisation
gb_wind_dispatch = (
    jan_2015.generators_t.p[gb_wind_names]
    .sum(axis=1)
)

# Implied curtailment
gb_wind_curtailment = (
    gb_wind_available - gb_wind_dispatch
).clip(lower=0)

# Remove tiny floating-point numerical noise
gb_wind_curtailment[
    gb_wind_curtailment < 1e-6
] = 0

# January totals
print(
    "GB wind available:",
    round(gb_wind_available.sum() / 1000, 2),
    "GWh"
)

print(
    "GB wind dispatched:",
    round(gb_wind_dispatch.sum() / 1000, 2),
    "GWh"
)

print(
    "GB wind curtailed:",
    round(gb_wind_curtailment.sum() / 1000, 2),
    "GWh"
)

print(
    "GB wind curtailment rate:",
    round(
        100
        * gb_wind_curtailment.sum()
        / gb_wind_available.sum(),
        2
    ),
    "%"
)

GB wind available: 5706.5 GWh
GB wind dispatched: 5705.27 GWh
GB wind curtailed: 1.24 GWh
GB wind curtailment rate: 0.02 %


In [38]:
# January 2015 wind results by carrier

wind_results_by_carrier = []

for carrier in wind_carriers:

    names = jan_2015.generators.index[
        jan_2015.generators["carrier"] == carrier
    ]

    capacity_mw = jan_2015.generators.loc[names, "p_nom"].sum()

    available = (
        jan_2015.generators_t.p_max_pu[names]
        .mul(jan_2015.generators.loc[names, "p_nom"], axis=1)
        .sum(axis=1)
    )

    dispatched = jan_2015.generators_t.p[names].sum(axis=1)

    curtailed = (available - dispatched).clip(lower=0)
    curtailed[curtailed < 1e-6] = 0

    available_gwh = available.sum() / 1000
    dispatched_gwh = dispatched.sum() / 1000
    curtailed_gwh = curtailed.sum() / 1000

    curtailment_pct = (
        100 * curtailed.sum() / available.sum()
        if available.sum() > 0
        else 0
    )

    wind_results_by_carrier.append({
        "carrier": carrier,
        "capacity_MW": capacity_mw,
        "available_GWh": available_gwh,
        "dispatched_GWh": dispatched_gwh,
        "curtailed_GWh": curtailed_gwh,
        "curtailment_pct": curtailment_pct
    })

wind_results_by_carrier = pd.DataFrame(wind_results_by_carrier)

print(
    wind_results_by_carrier.round({
        "capacity_MW": 2,
        "available_GWh": 2,
        "dispatched_GWh": 2,
        "curtailed_GWh": 3,
        "curtailment_pct": 3
    })
)

         carrier  capacity_MW  available_GWh  dispatched_GWh  curtailed_GWh  \
0   wind_onshore      7496.75        2921.15         2919.91          1.237   
1  wind_offshore      4039.40        1374.96         1374.96          0.000   
2  embedded_wind      3682.00        1410.39         1410.39          0.000   

   curtailment_pct  
0            0.042  
1            0.000  
2            0.000  


In [39]:
# Transmission line congestion statistics - January 2015

line_stats = []

for line in jan_2015.lines.index:

    s_nom = jan_2015.lines.at[line, "s_nom"]
    s_max_pu = jan_2015.lines.at[line, "s_max_pu"]

    limit = s_nom * s_max_pu

    # Skip lines with unusable/zero limits
    if limit <= 0:
        continue

    flow = jan_2015.lines_t.p0[line].abs()

    loading_pct = 100 * flow / limit

    line_stats.append({
        "line": line,
        "bus0": jan_2015.lines.at[line, "bus0"],
        "bus1": jan_2015.lines.at[line, "bus1"],
        "limit": limit,
        "max_loading_pct": loading_pct.max(),
        "avg_loading_pct": loading_pct.mean(),
        "hours_90": (loading_pct >= 90).sum(),
        "hours_95": (loading_pct >= 95).sum(),
        "hours_99": (loading_pct >= 99).sum(),
        "zero_r": jan_2015.lines.at[line, "r"] == 0
    })

line_stats = pd.DataFrame(line_stats)

line_stats = line_stats.sort_values(
    ["hours_99", "max_loading_pct"],
    ascending=False
)

print(
    line_stats.head(15).round({
        "limit": 2,
        "max_loading_pct": 2,
        "avg_loading_pct": 2
    })
)

   line                 bus0                 bus1   limit  max_loading_pct  \
16   16             Neilston              Deeside  2200.0           100.00   
1     1               Beauly             Errochty   132.0           100.00   
3     3               Beauly             Errochty   132.0           100.00   
62   62            Feckenham             Melksham  1970.0            88.70   
61   61            Feckenham             Melksham  1970.0            88.70   
57   57            Ratcliffe  Sundon/East Claydon  2100.0            73.76   
58   58            Ratcliffe  Sundon/East Claydon  2100.0            73.76   
0     0               Beauly            Peterhead   525.0            71.70   
2     2               Beauly            Peterhead   525.0            71.70   
75   75  Sundon/East Claydon               Keadby  2010.0            65.74   
76   76  Sundon/East Claydon               Keadby  2010.0            65.74   
38   38              Deeside            Feckenham  2400.0       

In [40]:
# Rank January 2015 onshore wind curtailment by bus

onshore_names = jan_2015.generators.index[
    jan_2015.generators["carrier"] == "wind_onshore"
]

# Hourly available generation for each onshore wind generator
onshore_available_by_gen = (
    jan_2015.generators_t.p_max_pu[onshore_names]
    .mul(
        jan_2015.generators.loc[onshore_names, "p_nom"],
        axis=1
    )
)

# Hourly dispatched generation
onshore_dispatch_by_gen = (
    jan_2015.generators_t.p[onshore_names]
)

# Curtailment by generator
onshore_curtailment_by_gen = (
    onshore_available_by_gen - onshore_dispatch_by_gen
).clip(lower=0)

onshore_curtailment_by_gen[
    onshore_curtailment_by_gen < 1e-6
] = 0

# Total curtailment by generator
curtailment_by_gen = onshore_curtailment_by_gen.sum()

# Map each generator to its bus
curtailment_by_bus = (
    pd.DataFrame({
        "bus": jan_2015.generators.loc[
            curtailment_by_gen.index, "bus"
        ],
        "curtailment_MWh": curtailment_by_gen.values
    })
    .groupby("bus")["curtailment_MWh"]
    .sum()
    .sort_values(ascending=False)
)

curtailment_by_bus = curtailment_by_bus[
    curtailment_by_bus > 1e-6
]

print("Onshore wind curtailment by bus:")
print((curtailment_by_bus / 1000).round(4).rename("curtailment_GWh"))

print(
    "\nTotal:",
    round(curtailment_by_bus.sum() / 1000, 4),
    "GWh"
)

Onshore wind curtailment by bus:
bus
Beauly    1.2371
Name: curtailment_GWh, dtype: float64

Total: 1.2371 GWh


In [41]:
# ---------------------------------------------------------
# Temporal overlap:
# Beauly wind curtailment vs Beauly-Errochty congestion
# ---------------------------------------------------------

# Onshore wind generators located at Beauly
beauly_wind_names = jan_2015.generators.index[
    (jan_2015.generators["carrier"] == "wind_onshore") &
    (jan_2015.generators["bus"] == "Beauly")
]

# Hourly available Beauly wind
beauly_wind_available = (
    jan_2015.generators_t.p_max_pu[beauly_wind_names]
    .mul(
        jan_2015.generators.loc[beauly_wind_names, "p_nom"],
        axis=1
    )
    .sum(axis=1)
)

# Hourly dispatched Beauly wind
beauly_wind_dispatch = (
    jan_2015.generators_t.p[beauly_wind_names]
    .sum(axis=1)
)

# Hourly Beauly wind curtailment
beauly_wind_curtailment = (
    beauly_wind_available - beauly_wind_dispatch
).clip(lower=0)

beauly_wind_curtailment[
    beauly_wind_curtailment < 1e-6
] = 0


# Beauly -> Errochty line 1 loading
line = "1"

line_limit = (
    jan_2015.lines.at[line, "s_nom"]
    * jan_2015.lines.at[line, "s_max_pu"]
)

beauly_errochty_loading = (
    jan_2015.lines_t.p0[line].abs()
    / line_limit
    * 100
)

# Define binding hours
binding_99 = beauly_errochty_loading >= 99

# Curtailment statistics
total_curtailment = beauly_wind_curtailment.sum()

curtailment_binding = (
    beauly_wind_curtailment[binding_99].sum()
)

curtailment_nonbinding = (
    beauly_wind_curtailment[~binding_99].sum()
)

curtailment_hours = (
    beauly_wind_curtailment > 1e-6
).sum()

curtailment_hours_binding = (
    (beauly_wind_curtailment > 1e-6)
    & binding_99
).sum()

print("Beauly wind curtailment:", round(total_curtailment, 2), "MWh")
print("Binding hours >=99%:", binding_99.sum())
print("Hours with curtailment:", curtailment_hours)
print("Curtailment hours while >=99%:", curtailment_hours_binding)

print(
    "Curtailment during >=99% loading:",
    round(curtailment_binding, 2),
    "MWh"
)

print(
    "Curtailment outside >=99% loading:",
    round(curtailment_nonbinding, 2),
    "MWh"
)

print(
    "Share of curtailment during >=99% loading:",
    round(
        100 * curtailment_binding / total_curtailment,
        2
    ),
    "%"
)

Beauly wind curtailment: 1237.11 MWh
Binding hours >=99%: 292
Hours with curtailment: 10
Curtailment hours while >=99%: 10
Curtailment during >=99% loading: 1237.11 MWh
Curtailment outside >=99% loading: 0.0 MWh
Share of curtailment during >=99% loading: 100.0 %


In [42]:
# Inspect the exact hours when Beauly wind was curtailed

curtailment_mask = beauly_wind_curtailment > 1e-6

event_table = pd.DataFrame({
    "wind_available_MW": beauly_wind_available,
    "wind_dispatch_MW": beauly_wind_dispatch,
    "wind_curtailment_MW": beauly_wind_curtailment,

    "line1_loading_pct":
        jan_2015.lines_t.p0["1"].abs()
        / (
            jan_2015.lines.at["1", "s_nom"]
            * jan_2015.lines.at["1", "s_max_pu"]
        ) * 100,

    "line3_loading_pct":
        jan_2015.lines_t.p0["3"].abs()
        / (
            jan_2015.lines.at["3", "s_nom"]
            * jan_2015.lines.at["3", "s_max_pu"]
        ) * 100,

    "line0_Beauly_Peterhead_MW":
        jan_2015.lines_t.p0["0"],

    "line2_Beauly_Peterhead_MW":
        jan_2015.lines_t.p0["2"]
})

event_table = event_table[curtailment_mask]

print(event_table.round(2))

                     wind_available_MW  wind_dispatch_MW  wind_curtailment_MW  \
snapshot                                                                        
2015-01-31 03:00:00             788.97            625.22               163.75   
2015-01-31 04:00:00             802.80            766.83                35.97   
2015-01-31 06:00:00             848.71            751.65                97.06   
2015-01-31 10:00:00             845.10            746.80                98.31   
2015-01-31 11:00:00             846.04            754.54                91.49   
2015-01-31 13:00:00             851.15            783.60                67.55   
2015-01-31 15:00:00             871.84            718.01               153.83   
2015-01-31 21:00:00             880.01            701.66               178.35   
2015-01-31 22:00:00             860.67            715.31               145.36   
2015-01-31 23:00:00             850.42            644.99               205.43   

                     line1_

In [43]:
import pypsa
import pandas as pd

# Load both solved models
jan_only = pypsa.Network(
    "../../resources/network/Research_2015_Jan_solved.nc"
)

jan_plus7 = pypsa.Network(
    "../../resources/network/Research_2015_JanPlus7_solved.nc"
)

print("Jan-only snapshots:", len(jan_only.snapshots))
print("Jan+7 snapshots:", len(jan_plus7.snapshots))


def beauly_metrics(n):

    # Beauly onshore wind generators
    names = n.generators.index[
        (n.generators["carrier"] == "wind_onshore") &
        (n.generators["bus"] == "Beauly")
    ]

    available = (
        n.generators_t.p_max_pu[names]
        .mul(n.generators.loc[names, "p_nom"], axis=1)
        .sum(axis=1)
    )

    dispatched = n.generators_t.p[names].sum(axis=1)

    curtailment = (available - dispatched).clip(lower=0)
    curtailment[curtailment < 1e-6] = 0

    # Beauly -> Errochty line 1
    line = "1"

    limit = (
        n.lines.at[line, "s_nom"]
        * n.lines.at[line, "s_max_pu"]
    )

    loading = (
        n.lines_t.p0[line].abs()
        / limit
        * 100
    )

    return pd.DataFrame({
        "available_MW": available,
        "dispatch_MW": dispatched,
        "curtailment_MW": curtailment,
        "line1_loading_pct": loading
    })


old = beauly_metrics(jan_only)
extended = beauly_metrics(jan_plus7)

# Compare ONLY 31 January
old_31 = old.loc["2015-01-31"]
extended_31 = extended.loc["2015-01-31"]

print("\n--- JANUARY-ONLY MODEL ---")
print("31 Jan curtailment:",
      round(old_31["curtailment_MW"].sum(), 2), "MWh")
print("Curtailment hours:",
      (old_31["curtailment_MW"] > 1e-6).sum())
print("Hours line >=99%:",
      (old_31["line1_loading_pct"] >= 99).sum())

print("\n--- JANUARY + 7 DAYS MODEL ---")
print("31 Jan curtailment:",
      round(extended_31["curtailment_MW"].sum(), 2), "MWh")
print("Curtailment hours:",
      (extended_31["curtailment_MW"] > 1e-6).sum())
print("Hours line >=99%:",
      (extended_31["line1_loading_pct"] >= 99).sum())

Jan-only snapshots: 744
Jan+7 snapshots: 912

--- JANUARY-ONLY MODEL ---
31 Jan curtailment: 1237.11 MWh
Curtailment hours: 10
Hours line >=99%: 24

--- JANUARY + 7 DAYS MODEL ---
31 Jan curtailment: 0.0 MWh
Curtailment hours: 0
Hours line >=99%: 21


In [44]:
# Compare TOTAL January Beauly curtailment
# between the two optimisation horizons

old_january = old.loc[
    "2015-01-01 00:00:00":"2015-01-31 23:00:00"
]

extended_january = extended.loc[
    "2015-01-01 00:00:00":"2015-01-31 23:00:00"
]

print(
    "Jan-only total Beauly curtailment:",
    round(old_january["curtailment_MW"].sum(), 2),
    "MWh"
)

print(
    "Jan+7 total Beauly curtailment:",
    round(extended_january["curtailment_MW"].sum(), 2),
    "MWh"
)

print(
    "Jan-only curtailment hours:",
    (old_january["curtailment_MW"] > 1e-6).sum()
)

print(
    "Jan+7 curtailment hours:",
    (extended_january["curtailment_MW"] > 1e-6).sum()
)

Jan-only total Beauly curtailment: 1237.11 MWh
Jan+7 total Beauly curtailment: 0.0 MWh
Jan-only curtailment hours: 10
Jan+7 curtailment hours: 0


In [45]:
import inspect

print(
    inspect.signature(
        jan_plus7.optimize.optimize_with_rolling_horizon
    )
)

(snapshots: 'Sequence | None' = None, horizon: 'int' = 100, overlap: 'int' = 0, **kwargs: 'Any') -> 'Network'


In [46]:
print(
    hasattr(
        jan_plus7.optimize,
        "optimize_with_rolling_horizon"
    )
)

True


In [47]:
import inspect

from scripts.solve import solve_network as sn

functions_to_check = [
    "configure_solver",
    "build_hydro_constraints_callback",
    "_build_neso_boundary_constraints_callback",
    "combine_extra_functionalities",
]

for name in functions_to_check:
    func = getattr(sn, name)

    print("\n", "=" * 70)
    print(name)
    print(inspect.signature(func))


configure_solver
(network, solver_name, solver_options, logger)

build_hydro_constraints_callback
(network, scenario_config: 'dict', large_hydro_storage_state: 'Optional[pd.Series]' = None, pumped_min_soc_override: 'Optional[float]' = None) -> 'Optional[Callable]'

_build_neso_boundary_constraints_callback
(network, scenario_config, logger)

combine_extra_functionalities
(*callbacks: 'Optional[Callable]') -> 'Optional[Callable]'


In [48]:
import inspect

print(inspect.getsource(sn.configure_solver))

def configure_solver(network, solver_name, solver_options, logger):
    """
    Configure optimization solver settings.
    
    Parameters
    ----------
    network : pypsa.Network
        Network to optimize
    solver_name : str
        Solver to use ('gurobi', 'highs', 'glpk', 'cplex')
    solver_options : dict
        Solver-specific options
    logger : logging.Logger
        Logger instance
    """
    logger.info(f"Configuring solver: {solver_name}")
    logger.info(f"Solver options: {solver_options}")
    
    # Validate solver availability
    available_solvers = ['gurobi', 'highs', 'glpk', 'cplex']
    if solver_name not in available_solvers:
        logger.warning(f"Unknown solver '{solver_name}', falling back to 'highs'")
        solver_name = 'highs'
    
    return solver_name, solver_options



In [49]:
print(inspect.getsource(sn.combine_extra_functionalities))

def combine_extra_functionalities(*callbacks: Optional[Callable]) -> Optional[Callable]:
    """Combine multiple PyPSA extra_functionality callbacks into one."""
    active_callbacks = [callback for callback in callbacks if callback is not None]
    if not active_callbacks:
        return None

    def combined(network, snapshots):
        for callback in active_callbacks:
            callback(network, snapshots)

    return combined



In [50]:
preprocessing_functions = [
    "validate_network_costs",
    "apply_transmission_relaxation",
    "apply_line_rating_overrides",
    "apply_outage_schedule",
    "improve_numerical_conditioning",
    "apply_load_shedding_limits",
]

for name in preprocessing_functions:
    func = getattr(sn, name)

    print("\n" + "=" * 70)
    print(name)
    print(inspect.signature(func))


validate_network_costs
(network, logger)

apply_transmission_relaxation
(network, scenario_config, logger)

apply_line_rating_overrides
(network, scenario_config, logger)

apply_outage_schedule
(network, scenario_config, logger)

improve_numerical_conditioning
(network, logger)

apply_load_shedding_limits
(network, logger)


In [1]:
import pypsa
import pandas as pd

# Full-horizon reference
reference = pypsa.Network(
    "../../resources/network/Research_2015_JanPlus7_solved.nc"
)

# New rolling-horizon result
rolling = pypsa.Network(
    "../../resources/network/Research_2015_JanPlus7_rolling_solved.nc"
)


def january_metrics(n):

    jan = n.snapshots[
        (n.snapshots >= "2015-01-01 00:00:00") &
        (n.snapshots <= "2015-01-31 23:00:00")
    ]

    # -----------------------------------
    # Beauly onshore wind
    # -----------------------------------

    wind = n.generators.index[
        (n.generators["carrier"] == "wind_onshore") &
        (n.generators["bus"] == "Beauly")
    ]

    available = (
        n.generators_t.p_max_pu.loc[jan, wind]
        .mul(n.generators.loc[wind, "p_nom"], axis=1)
        .sum(axis=1)
    )

    dispatched = (
        n.generators_t.p.loc[jan, wind]
        .sum(axis=1)
    )

    curtailed = (
        available - dispatched
    ).clip(lower=0)

    curtailed[curtailed < 1e-6] = 0


    # -----------------------------------
    # Beauly -> Errochty line 1
    # -----------------------------------

    line = "1"

    limit = (
        n.lines.at[line, "s_nom"]
        * n.lines.at[line, "s_max_pu"]
    )

    loading = (
        n.lines_t.p0.loc[jan, line].abs()
        / limit
        * 100
    )


    # -----------------------------------
    # Load shedding
    # -----------------------------------

    load_shedding = n.generators.index[
        n.generators["carrier"] == "load_shedding"
    ]

    load_shedding_mwh = (
        n.generators_t.p.loc[jan, load_shedding]
        .sum()
        .sum()
    )


    return {
        "Beauly curtailment MWh":
            curtailed.sum(),

        "Beauly curtailment hours":
            (curtailed > 1e-6).sum(),

        "Beauly-Errochty >=99% hours":
            (loading >= 99).sum(),

        "Load shedding MWh":
            load_shedding_mwh
    }


comparison = pd.DataFrame({
    "Full Jan+7": january_metrics(reference),
    "Rolling": january_metrics(rolling)
})

print(comparison.round(2))

INFO:pypsa.network.io:New version 1.2.4 available! (Current: 1.0.7)
INFO:pypsa.network.io:Imported network 'Research_2015_JanPlus7 (Full)' has buses, carriers, generators, lines, links, loads, storage_units, sub_networks
INFO:pypsa.network.io:New version 1.2.4 available! (Current: 1.0.7)
INFO:pypsa.network.io:Imported network 'Research_2015_JanPlus7 (Full)' has buses, carriers, generators, lines, links, loads, storage_units, sub_networks


                             Full Jan+7  Rolling
Beauly curtailment MWh              0.0  3057.61
Beauly curtailment hours            0.0    32.00
Beauly-Errochty >=99% hours       247.0   435.00
Load shedding MWh                   0.0     0.00
